# GPR Ballast Fouling Classification

This notebook trains a Random Forest classifier on the synthetic GPR feature dataset.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import SelectFromModel

%matplotlib inline

## 2. Load Data

In [ ]:
# Path to the features.csv generated by the pipeline
# UPDATE THIS PATH TO YOUR LATEST RUN
DATA_PATH = r'D:\Codigo\Synth-Data\20251208_113403\features.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape}")
df.head()

## 3. Preprocessing
- Drop ID columns (`sample_id`)
- Drop leakage/regression targets (`label_pvc`)
- Encode Target (`label_FI_class`)

In [ ]:
# Define Target and Drop irrelevant columns
target_col = 'label_FI_class'
drop_cols = ['label_pvc', 'sample_id', 'meta_sample_id', 'filename'] 
# Note: 'meta_*' columns should already be gone, but good to be safe

# X (Features) and y (Target)
X = df.drop(columns=[col for col in drop_cols if col in df.columns] + [target_col])
y = df[target_col]

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_)
print("Feature Matrix shape:", X.shape)

## 4. Feature Selection
Since we have >500 features and 500 samples, we use a preliminary Random Forest to select the most important features.

In [ ]:
print("Selecting features...")
sel_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
sel_clf.fit(X, y_encoded)

# Select features with importance > mean importance
selector = SelectFromModel(sel_clf, threshold='1.25*mean')
selector.fit(X, y_encoded)

X_selected = selector.transform(X)
selected_feats = X.columns[selector.get_support()]

print(f"Selected {X_selected.shape[1]} features out of {X.shape[1]}")
print("Top 10 Examples:", list(selected_feats[:10]))

## 5. Model Training & Evaluation

In [ ]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

# Train Final Classifer
clf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

## 6. Visualization

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Feature Importance Plot
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 8))
plt.title("Top 20 Feature Importances")
plt.barh(range(20), importances[indices[:20]], align="center")
plt.yticks(range(20), selected_feats[indices[:20]])
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()